# Credit Card Fraud Detection — Simplified Deep Learning Notebook

**Goal:** Detect fraudulent transactions using two tabular deep-learning models:
1. FT-Transformer
2. TabNet

This notebook keeps the workflow short: load data → split → scale → train/tune → evaluate → compare.

**Dataset:** `creditcard.csv` from the Kaggle Credit Card Fraud Detection dataset. Upload the CSV when prompted.

**Important:** This is a simplified experimental version. It uses a small Optuna search (not separate Hyperband and Genetic Algorithm experiments). It is intended to be easy to run and debug first. For a research-paper comparison, use identical splits, multiple seeds, and document the tuning budget.

## 1. Install required packages

In [ ]:
!pip -q install pytorch-tabnet optuna imbalanced-learn

## 2. Imports and GPU check

In [ ]:
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import optuna

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
from pytorch_tabnet.tab_model import TabNetClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Upload and read the dataset

In [ ]:
uploaded = files.upload()  # Upload creditcard.csv
DATA_PATH = next(iter(uploaded))
df = pd.read_csv(DATA_PATH)

assert "Class" in df.columns, "Dataset must contain a 'Class' column."
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates().reset_index(drop=True)

X = df.drop(columns=["Class"]).select_dtypes(include=[np.number])
y = df["Class"].astype(int)

print("Dataset shape:", df.shape)
print("Class counts:")
print(y.value_counts())
print("Fraud rate: {:.4f}%".format(100 * y.mean()))

## 4. Split data and scale it

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    stratify=y_train_full, random_state=SEED
)

# Fill missing values using training medians only
medians = X_train.median()
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

y_train = y_train.to_numpy().astype("int64")
y_val = y_val.to_numpy().astype("int64")
y_test = y_test.to_numpy().astype("int64")

print("Train:", X_train.shape, " Validation:", X_val.shape, " Test:", X_test.shape)

## 5. Handle class imbalance

For a quick and reliable first run, use class-weighted loss for FT-Transformer and TabNet's class weights are not explicitly altered here. This avoids the extra runtime and memory cost of SMOTE.

The validation and test sets remain untouched.

## 6. Define a compact FT-Transformer

In [ ]:
class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=32, n_heads=4, dropout=0.2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=d_token * 2,
            dropout=dropout, batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, 1)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)
        cls = self.cls.expand(x.size(0), -1, -1)
        z = torch.cat([cls, tokens], dim=1)
        z = self.encoder(z)
        return self.head(self.norm(z[:, 0])).squeeze(1)

## 7. FT-Transformer training helper

In [ ]:
def train_ft(params, epochs=4):
    model = FTTransformer(
        n_features=X_train.shape[1],
        d_token=params["d_token"],
        n_heads=params["n_heads"],
        dropout=params["dropout"]
    ).to(DEVICE)

    # Weight the positive class to account for rare fraud examples
    pos_weight = torch.tensor(
        [(len(y_train) - y_train.sum()) / max(y_train.sum(), 1)],
        dtype=torch.float32, device=DEVICE
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["lr"], weight_decay=1e-5
    )

    batch_size = params["batch_size"]
    for epoch in range(epochs):
        model.train()
        order = np.random.permutation(len(X_train))
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            xb = torch.tensor(X_train[idx], device=DEVICE)
            yb = torch.tensor(y_train[idx], dtype=torch.float32, device=DEVICE)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    return model


@torch.no_grad()
def predict_ft(model, X_data, batch_size=4096):
    model.eval()
    predictions = []
    for start in range(0, len(X_data), batch_size):
        xb = torch.tensor(X_data[start:start + batch_size], device=DEVICE)
        predictions.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(predictions)

## 8. Tune FT-Transformer with a small Optuna search

In [ ]:
def ft_objective(trial):
    params = {
        "d_token": trial.suggest_categorical("d_token", [16, 32, 64]),
        "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.35),
        "lr": trial.suggest_float("lr", 1e-4, 2e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512])
    }
    model = train_ft(params, epochs=3)
    val_probs = predict_ft(model, X_val)
    return average_precision_score(y_val, val_probs)

study_ft = optuna.create_study(direction="maximize")
study_ft.optimize(ft_objective, n_trials=3)

print("Best FT validation PR-AUC:", study_ft.best_value)
print("Best FT parameters:", study_ft.best_params)

## 9. Tune and train TabNet

In [ ]:
def tabnet_objective(trial):
    model = TabNetClassifier(
        n_d=trial.suggest_categorical("n_d", [8, 16, 24]),
        n_a=trial.suggest_categorical("n_a", [8, 16, 24]),
        n_steps=trial.suggest_int("n_steps", 3, 5),
        optimizer_fn=torch.optim.Adam,
        optimizer_params={
            "lr": trial.suggest_float("lr", 1e-3, 1e-2, log=True)
        },
        seed=SEED,
        verbose=0,
        device_name=DEVICE
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_name=["valid"],
        eval_metric=["auc"],
        max_epochs=5,
        patience=2,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )
    val_probs = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, val_probs)

study_tabnet = optuna.create_study(direction="maximize")
study_tabnet.optimize(tabnet_objective, n_trials=3)

print("Best TabNet validation PR-AUC:", study_tabnet.best_value)
print("Best TabNet parameters:", study_tabnet.best_params)

## 10. Train final models using the selected parameters

In [ ]:
ft_params = study_ft.best_params
tab_params = study_tabnet.best_params

start = time.time()
final_ft = train_ft(ft_params, epochs=8)
ft_train_seconds = time.time() - start

start = time.time()
final_tabnet = TabNetClassifier(
    n_d=tab_params["n_d"],
    n_a=tab_params["n_a"],
    n_steps=tab_params["n_steps"],
    optimizer_fn=torch.optim.Adam,
    optimizer_params={"lr": tab_params["lr"]},
    seed=SEED,
    verbose=0,
    device_name=DEVICE
)
final_tabnet.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=12,
    patience=3,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)
tabnet_train_seconds = time.time() - start

print(f"FT-Transformer training time: {ft_train_seconds/60:.2f} min")
print(f"TabNet training time: {tabnet_train_seconds/60:.2f} min")

## 11. Choose thresholds using validation data only

In [ ]:
def choose_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if len(thresholds) == 0:
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-12
    )
    return float(thresholds[np.argmax(f1)])

ft_val_probs = predict_ft(final_ft, X_val)
tab_val_probs = final_tabnet.predict_proba(X_val)[:, 1]

ft_threshold = choose_threshold(y_val, ft_val_probs)
tab_threshold = choose_threshold(y_val, tab_val_probs)

print("FT threshold:", ft_threshold)
print("TabNet threshold:", tab_threshold)

## 12. Final test evaluation

In [ ]:
def evaluate(name, y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall / Detection Rate": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probs),
        "PR-AUC": average_precision_score(y_true, probs)
    }

ft_test_probs = predict_ft(final_ft, X_test)
tab_test_probs = final_tabnet.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    evaluate("FT-Transformer", y_test, ft_test_probs, ft_threshold),
    evaluate("TabNet", y_test, tab_test_probs, tab_threshold)
])
display(results.round(4))

## 13. Confusion matrices and classification reports

In [ ]:
for name, probs, threshold in [
    ("FT-Transformer", ft_test_probs, ft_threshold),
    ("TabNet", tab_test_probs, tab_threshold)
]:
    preds = (probs >= threshold).astype(int)
    print("\\n", name)
    print(classification_report(y_test, preds, digits=4, zero_division=0))

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 14. ROC and Precision-Recall curves

In [ ]:
plt.figure(figsize=(6, 5))
for name, probs in [("FT-Transformer", ft_test_probs), ("TabNet", tab_test_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")
plt.plot([0, 1], [0, 1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(6, 5))
for name, probs in [("FT-Transformer", ft_test_probs), ("TabNet", tab_test_probs)]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()
plt.show()

## 15. Save results and models

In [ ]:
results.to_csv("fraud_detection_results.csv", index=False)
torch.save(final_ft.state_dict(), "ft_transformer_fraud.pth")
final_tabnet.save_model("tabnet_fraud_model")
print("Saved results and model files.")
files.download("fraud_detection_results.csv")
files.download("ft_transformer_fraud.pth")